# Embeddings Tutorial - Complete Guide

This notebook teaches you everything about text embeddings for RAG:

## What You'll Learn:
1. What are embeddings and why they matter
2. Different embedding models (local vs cloud)
3. Model comparison (size, speed, quality)
4. How to choose the right model
5. Impact on retrieval quality

## Why Embeddings Matter:
- Embeddings = numerical representations of text
- Similar text → Similar vectors
- Enable semantic search (meaning, not just keywords)
- Quality of embeddings directly impacts RAG accuracy

In [ ]:
import sys
sys.path.append('..')

from vectordb.embeddings import (
    embed_chunks,
    get_provider,
    EmbeddingConfig,
    create_embed_function,
    list_local_models,
    get_embedding_statistics
)
from vectordb.chunking import chunk_text, ChunkingConfig, ChunkingStrategy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

## Part 1: Understanding Embeddings

Let's see what an embedding actually looks like:

In [ ]:
# Create simple text examples
texts = [
    "Machine learning is a subset of artificial intelligence.",
    "AI and ML are transforming technology.",
    "I love eating pizza for dinner."
]

# Create embedding function (using default local model)
embed_fn = create_embed_function()

# Generate embeddings
print("Text → Embedding Vectors:\n")
for text in texts:
    vector = embed_fn(text)
    print(f"Text: {text}")
    print(f"Vector (first 10 dims): {vector[:10]}")
    print(f"Vector dimensions: {len(vector)}")
    print(f"Vector type: {type(vector[0])}\n")

## Part 2: Semantic Similarity

Similar texts have similar vectors (measured by cosine similarity):

In [ ]:
def cosine_similarity(v1, v2):
    """Calculate cosine similarity between two vectors."""
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = sum(a * a for a in v1) ** 0.5
    norm2 = sum(b * b for b in v2) ** 0.5
    return dot / (norm1 * norm2) if norm1 and norm2 else 0.0

# Get embeddings
embeddings = [embed_fn(text) for text in texts]

# Calculate similarities
print("SEMANTIC SIMILARITY MATRIX")
print("="*60)
print(f"{'':40} {'Text 1':>8} {'Text 2':>8} {'Text 3':>8}")
print("="*60)

for i, text1 in enumerate(texts):
    similarities = []
    for j, text2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        similarities.append(f"{sim:.3f}")
    
    label = f"Text {i+1}: {text1[:30]}..."
    print(f"{label:40} {similarities[0]:>8} {similarities[1]:>8} {similarities[2]:>8}")

print("\n💡 Observations:")
print("- Texts 1 & 2 (both about ML/AI) have HIGH similarity (~0.7-0.9)")
print("- Text 3 (about pizza) has LOW similarity to 1 & 2 (~0.1-0.3)")
print("- Each text is identical to itself (similarity = 1.0)")

## Part 3: Available Embedding Models

Let's see what local models are available:

In [ ]:
models = list_local_models()

print("AVAILABLE LOCAL EMBEDDING MODELS")
print("="*60)
print(f"{'Model Name':40} {'Dimensions':>12}")
print("="*60)

for model, dims in models.items():
    print(f"{model:40} {dims:>12}")

print("\n📊 Model Comparison:")
print("\n🚀 Fast & Small (384 dims):")
print("  - all-MiniLM-L6-v2 (default, RECOMMENDED for most cases)")
print("  - BAAI/bge-small-en-v1.5")
print("  - intfloat/e5-small-v2")

print("\n⚖️ Balanced (768 dims):")
print("  - all-mpnet-base-v2 (best quality/speed)")
print("  - BAAI/bge-base-en-v1.5")
print("  - intfloat/e5-base-v2")

print("\n🎯 Large & Accurate (1024 dims):")
print("  - BAAI/bge-m3 (multilingual, highest quality)")

## Part 4: Model Comparison - Speed vs Quality

Let's benchmark different models:

In [ ]:
# Sample text to embed
sample_texts = [
    "Machine learning algorithms can learn from data.",
    "Deep neural networks are powerful models.",
    "Natural language processing enables text understanding.",
    "Computer vision helps machines see the world.",
    "Reinforcement learning trains agents through rewards."
] * 10  # 50 texts total

# Models to test (start with small models for speed)
test_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
]

results = []

print("BENCHMARKING MODELS...\n")

for model_name in test_models:
    print(f"Testing {model_name}...")
    
    config = EmbeddingConfig(model=model_name)
    provider = get_provider("sentence-transformers", config)
    
    # Warm-up
    _ = provider.embed_texts([sample_texts[0]])
    
    # Benchmark
    start = time.time()
    embeddings = provider.embed_texts(sample_texts)
    elapsed = time.time() - start
    
    results.append({
        "Model": model_name,
        "Dimensions": provider.dimensions,
        "Time (s)": f"{elapsed:.2f}",
        "Speed (texts/s)": f"{len(sample_texts)/elapsed:.1f}",
        "Time per text (ms)": f"{1000*elapsed/len(sample_texts):.1f}"
    })
    
    print(f"  ✓ Completed in {elapsed:.2f}s\n")

df = pd.DataFrame(results)
print("\nBENCHMARK RESULTS")
print("="*80)
print(df.to_string(index=False))

print("\n💡 Key Takeaways:")
print("- Smaller models (384 dims) are FASTER")
print("- Larger models (768+ dims) are more ACCURATE but slower")
print("- For most RAG use cases, all-MiniLM-L6-v2 is perfect!")

## Part 5: Embedding Chunks

Now let's embed actual document chunks:

In [ ]:
# Sample document
doc_text = """
# Python Programming

Python is a high-level programming language known for its simplicity and readability.
It's widely used in data science, machine learning, web development, and automation.

## Key Features

- Easy to learn and use
- Large standard library
- Strong community support
- Cross-platform compatibility

## Applications

Python is used in many fields:
- Data Science: pandas, numpy, scikit-learn
- Machine Learning: TensorFlow, PyTorch
- Web Development: Django, Flask
- Automation: Selenium, PyAutoGUI
"""

# Chunk the document
chunk_config = ChunkingConfig(
    strategy=ChunkingStrategy.RECURSIVE,
    chunk_size=200,
    chunk_overlap=20
)

chunks = chunk_text(doc_text, "python_guide.md", chunk_config)
print(f"Created {len(chunks)} chunks\n")

# Embed with default model
embed_config = EmbeddingConfig(model="all-MiniLM-L6-v2")
provider = get_provider("sentence-transformers", embed_config)

embeddings = embed_chunks(chunks, provider)
stats = get_embedding_statistics(embeddings)

print("EMBEDDING STATISTICS")
print("="*60)
print(f"Total embeddings: {stats['total_embeddings']}")
print(f"Unique sources: {stats['unique_sources']}")
print(f"Model: {stats['model']}")
print(f"Dimensions: {stats['dimensions']}")

print("\nSample embedding:")
print(f"Chunk text: {embeddings[0].chunk.content[:100]}...")
print(f"Vector (first 10): {list(embeddings[0].vector[:10])}")

## Part 6: Comparing Embedding Models on Same Text

Let's see how different models embed the same text:

In [ ]:
test_query = "machine learning and data science"

models_to_compare = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
]

print(f"Query: '{test_query}'\n")
print("="*80)

for model_name in models_to_compare:
    config = EmbeddingConfig(model=model_name)
    provider = get_provider("sentence-transformers", config)
    
    # Embed query
    query_vector = provider.embed_texts([test_query])[0]
    
    # Embed chunks
    chunk_embeddings = embed_chunks(chunks, provider)
    
    # Find most similar chunk
    similarities = []
    for emb in chunk_embeddings:
        sim = cosine_similarity(query_vector, list(emb.vector))
        similarities.append((sim, emb.chunk.content))
    
    similarities.sort(reverse=True)
    
    print(f"\nModel: {model_name} ({provider.dimensions} dims)")
    print(f"Top match (similarity: {similarities[0][0]:.3f}):")
    print(f"  {similarities[0][1][:150]}...")
    print("\n" + "="*80)

## Part 7: Visualizing Embeddings

Let's visualize how embeddings cluster similar content:

In [ ]:
from sklearn.decomposition import PCA

# Create diverse texts
diverse_texts = [
    # ML/AI cluster
    "Machine learning models learn from data",
    "Neural networks process information",
    "Deep learning achieves high accuracy",
    
    # Programming cluster
    "Python is a programming language",
    "Write code to solve problems",
    "Functions and classes in software",
    
    # Food cluster
    "Pizza is delicious Italian food",
    "Cooking pasta requires boiling water",
    "Fresh vegetables make healthy meals"
]

labels = [
    "ML", "ML", "ML",
    "Code", "Code", "Code",
    "Food", "Food", "Food"
]

# Embed
embed_fn = create_embed_function()
vectors = [embed_fn(text) for text in diverse_texts]

# Reduce to 2D using PCA
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

# Plot
plt.figure(figsize=(10, 8))

colors = {'ML': 'red', 'Code': 'blue', 'Food': 'green'}
for i, (x, y) in enumerate(vectors_2d):
    label = labels[i]
    plt.scatter(x, y, c=colors[label], s=200, alpha=0.6, edgecolors='black')
    plt.annotate(f"{i+1}", (x, y), ha='center', va='center', fontsize=10, fontweight='bold')

# Legend
for label, color in colors.items():
    plt.scatter([], [], c=color, s=200, alpha=0.6, label=label)

plt.xlabel('PCA Component 1', fontsize=12)
plt.ylabel('PCA Component 2', fontsize=12)
plt.title('Embedding Space Visualization (384D → 2D)', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nText Labels:")
for i, text in enumerate(diverse_texts, 1):
    print(f"{i}. [{labels[i-1]}] {text}")

print("\n💡 Observations:")
print("- Similar topics cluster together")
print("- ML, Code, and Food form distinct groups")
print("- This is how semantic search works!")

## Part 8: Model Selection Guide

### When to use each model:

| Model | Use Case | Pros | Cons |
|-------|----------|------|------|
| **all-MiniLM-L6-v2** | General purpose, RECOMMENDED | Fast, good quality, small | Lower accuracy than large models |
| **all-mpnet-base-v2** | Higher quality needed | Best quality/speed balance | Slower than MiniLM |
| **BAAI/bge-small-en-v1.5** | Production RAG systems | Great accuracy, optimized | English only |
| **BAAI/bge-base-en-v1.5** | Best English quality | Excellent accuracy | Slower |
| **BAAI/bge-m3** | Multilingual documents | Supports 100+ languages | Largest, slowest |
| **OpenAI text-embedding-3-small** | Cloud, simple setup | No local compute needed | Costs money, API calls |

### Decision Tree:

```
Do you need multilingual support?
├─ YES → BAAI/bge-m3
└─ NO → Is speed critical?
    ├─ YES → all-MiniLM-L6-v2 (RECOMMENDED)
    └─ NO → Do you need highest quality?
        ├─ YES → all-mpnet-base-v2 or BAAI/bge-base-en-v1.5
        └─ NO → all-MiniLM-L6-v2 (RECOMMENDED)
```

## Part 9: Impact on Retrieval Quality

Let's see how model choice affects retrieval:

In [ ]:
# Test queries with different semantic meanings
test_queries = [
    "What is ML?",  # Direct match
    "Tell me about artificial intelligence",  # Semantic match
    "How to code in Python?",  # Different topic
]

models = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}")
    
    for model_name in models:
        config = EmbeddingConfig(model=model_name)
        provider = get_provider("sentence-transformers", config)
        
        # Embed query
        query_vec = provider.embed_texts([query])[0]
        
        # Embed chunks
        chunk_embs = embed_chunks(chunks, provider)
        
        # Find best match
        best_sim = 0
        best_chunk = None
        
        for emb in chunk_embs:
            sim = cosine_similarity(query_vec, list(emb.vector))
            if sim > best_sim:
                best_sim = sim
                best_chunk = emb.chunk.content
        
        print(f"\n{model_name}:")
        print(f"  Similarity: {best_sim:.3f}")
        print(f"  Match: {best_chunk[:100]}...")

## Part 10: Your Turn - Experiment!

Try different models on your own text:

In [ ]:
# YOUR EXPERIMENT

your_text = """
Paste your text here to experiment with different embedding models...
"""

your_query = "your search query"

# Try different models
model_to_test = "all-MiniLM-L6-v2"  # Change this

config = EmbeddingConfig(model=model_to_test)
provider = get_provider("sentence-transformers", config)

# Chunk and embed
chunks = chunk_text(your_text, "my_doc.md", ChunkingConfig())
embeddings = embed_chunks(chunks, provider)

print(f"Model: {model_to_test}")
print(f"Embeddings created: {len(embeddings)}")
print(f"Dimensions: {provider.dimensions}")

## Summary

### What You Learned:

✅ **What embeddings are**: Numerical representations of text  
✅ **Semantic similarity**: How to measure text similarity  
✅ **Model comparison**: Speed vs quality tradeoffs  
✅ **Model selection**: Choosing the right model for your use case  
✅ **Visualization**: Understanding the embedding space  

### Key Recommendations:

1. **Start with `all-MiniLM-L6-v2`** - best default choice
2. **Upgrade to `all-mpnet-base-v2`** if you need better quality
3. **Use `BAAI/bge-m3`** for multilingual documents
4. **Always normalize** embeddings for cosine similarity
5. **Batch processing** for efficiency (100+ texts at once)

### Next Steps:

1. **Next Notebook**: `3_vector_store_tutorial.ipynb` - Store and search embeddings
2. **Experiment**: Try different models on your documents
3. **Benchmark**: Test quality vs speed on your specific use case

### Quick Reference:

```python
from vectordb.embeddings import embed_chunks, get_provider, EmbeddingConfig

# Recommended for most cases
config = EmbeddingConfig(model="all-MiniLM-L6-v2")
provider = get_provider("sentence-transformers", config)
embeddings = embed_chunks(chunks, provider)
```